### Summarizing Mani Mama Lecture

* The youtube transcripts were not really good and there were a lot of mistakes. That is why we had to download the video and use openai-whisper library to get it transcribed. Use the transcribe.py to take the MP4 files and output the transcript into a text file.

In [2]:
from langchain_ollama import ChatOllama
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_chroma import Chroma
import textwrap
import chromadb

DB_DIR = "./chromadb"
# Initialize your Chroma vector store
chroma_client = chromadb.PersistentClient(path=DB_DIR)

vector_store = Chroma(collection_name="mani_mama_collection", client=chroma_client)


def query_after_getting_matched_documents(user_query, ollama_model_name="granite4.1:3b"):
    # Create a retriever from the vector store getting top 10 similar documents
    retriever = vector_store.as_retriever(collection_name="mani_mama_collection", search_type="similarity", search_kwargs={"k": 5})

    llm = ChatOllama(model=ollama_model_name, base_url=None)
    # ConversationalRetrievalChain wraps the LLM + retriever
    chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, return_source_documents=True)

    result = chain.invoke({"question": user_query, "chat_history":[]})
    print(textwrap.fill(result["answer"], width=70))
    matching_docs = result["source_documents"]
    return matching_docs

In [ ]:
query = "Why is shama very important before vedantic meditation?" 
matching_docs = query_after_getting_matched_documents(query)
print("Matching document IDs:")
for doc in matching_docs:
    print('----------------------')
    print(textwrap.fill(str(doc.id), width=70))
    print(textwrap.fill(str(doc.page_content), width=70))
    print('----------------------')

Shama (sense control) is very important before Vedantic meditation
because, despite our ability to regulate and limit sense perceptions
through controlling the sense organs, it is impossible to completely
eliminate all external stimuli that enter the mind. These objects can
trigger various negative emotions such as raga (attachment), advesha
(aversion), krodha (anger), loba (greed), moha (delusion), etc., which
disrupt our mental state and prevent us from attaining a calm, focused
mind necessary for Vedantic meditation. By practicing shama, we reduce
the influence of these unwanted thoughts and perceptions, allowing the
mind to become undisturbed and better prepared for deeper meditative
states aligned with Vedanta teachings.
Matching document IDs:
----------------------
videos/006.txt[0:40:00 - 0:41:00]
----------------------
----------------------
videos/014.txt[0:09:00 - 0:10:00]
----------------------
----------------------
videos/014.txt[0:14:00 - 0:15:00]
----------------------
-

In [17]:
query = "What makes one an adhikari for self knowledge?" 
matching_docs = query_after_getting_matched_documents(query)
print("Matching document IDs:")
for doc in matching_docs:
    print('----------------------')
    print(textwrap.fill(str(doc.id), width=70))
    print('----------------------')


According to the text, being called an **Adhikari** (the eligible
person) for self-knowledge requires possessing certain qualities known
as the *Sadhana Chaturthaka Sampatti*. These include:  1. **Viveka** –
discriminative ability or discernment, which allows one to distinguish
between what is truly the Self (*Atma*) and that which is not
(*Anatma*). 2. **Vairagya** – dispassion or detachment from worldly
desires and attachments. 3. **Shatka Sampati** (part of the four
qualifications) includes *Mumukshutvam* – a sincere craving for
liberation (*Moksha*) through self-knowledge.  Thus, an Adhikari must
exhibit these qualities: intellectual discernment, emotional
detachment, and a deep desire to attain Moksha. The text suggests that
without acquiring these characteristics—through practices such as
meditation, study of the Vedas (like Bhagavad Gita), and cultivation
of Vairagya—one cannot be eligible for self-knowledge or the deeper
teachings of Vedanta. Essentially, becoming an Adhikari i

In [16]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.documents import Document

ollama_model_name="granite4.1:3b"
llm = ChatOllama(model=ollama_model_name, base_url=None)

collection = chroma_client.get_or_create_collection(name="mani_mama_collection")

# Call the vector database for matching docs
results = collection.get(
    where={'chapter': {"$eq": "Chapter-02"}}
)

docs = []

m = results["documents"]
print("Got " + str(len(m)) + " matching documents")
for x in m:
    docs.append(Document(m[0]))
    

prompt_template = ChatPromptTemplate.from_messages(
    [("system", "Act as a critical researcher. \\n Write a 500 word summary of the following:\\n{context}")]
)

chain = create_stuff_documents_chain(llm, prompt_template)

ans = chain.invoke({'context': docs})
print(textwrap.fill(ans, width=80))

Got 574 matching documents
The tenth sloka of the Bhagavad‑Gita (Bhag. II.10) states that Arjuna was
confused and overwhelmed by his duties, and therefore he could not see or
understand the spiritual truth before him. In other words, due to his emotional
turmoil and lack of clarity, Arjuna’s vision was obscured so that even learned
sages (panditah) could not discern what Krishna was teaching.  **Key points from
this verse:**  1. **Arjuna’s Condition:**      - He was “bewildered”
(asuravijñāna – ignorance of the demonic nature).      - His mind was clouded by
attachment and fear, preventing him from grasping the higher purpose of action.
2. **Resulting Impairment:**      - Because of this inner confusion, Arjuna
could not see or understand Krishna’s teachings as clearly as a sage would.
- The verse emphasizes that it is not merely his physical presence on the
battlefield that hinders comprehension; it is his mental state (asuravijñāna)
and resulting lack of discernment.  3. **Implicatio